# Q2 · Modelagem Preditiva de Prevenção
Score diário de risco regulatório (0–100) para clientes em 1ª instância. Clientes com score ≥ 70 são enfileirados para interceptação proativa antes de abrirem RDR.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import shap

plt.rcParams['figure.dpi'] = 120
SEED = 42
print('Bibliotecas carregadas.')

## 1 · Carregar e preparar dados

In [ ]:
aba1 = pd.read_csv('../outputs/aba1_limpa.csv')
aba1['INCOMING_DTTM'] = pd.to_datetime(aba1['INCOMING_DTTM'], format='mixed')
aba2 = pd.read_csv('../outputs/aba2_limpa.csv')
aba2['RDR_EXTERNAL_CREATE_DTTM'] = pd.to_datetime(aba2['RDR_EXTERNAL_CREATE_DTTM'], format='mixed')
aba2['RDR_OUTGOING_DTTM']        = pd.to_datetime(aba2['RDR_OUTGOING_DTTM'], format='mixed', errors='coerce')

# Período sobreponível: set–nov 2025
aba1 = aba1[aba1['INCOMING_DTTM'] >= '2025-09-01'].copy()

print(f'Aba 1 (set–nov): {aba1.shape[0]:,} registros')
print(f'Aba 2:           {aba2.shape[0]:,} registros')

## 2 · Criar variável target: abriu RDR nas 48h seguintes?

In [ ]:
id_col_1 = 'ID do usuário'
id_col_2 = 'ID do usuario'

aba1[id_col_1] = pd.to_numeric(aba1[id_col_1], errors='coerce')
aba2[id_col_2] = pd.to_numeric(aba2[id_col_2], errors='coerce')

aba2_rdr = aba2[['ID do usuario', 'RDR_EXTERNAL_CREATE_DTTM']].dropna().copy()
aba2_rdr.columns = ['id', 'rdr_dt']

aba1_work = aba1.rename(columns={id_col_1: 'id'}).copy()
aba1_work = aba1_work.dropna(subset=['id'])
aba1_work['id'] = aba1_work['id'].astype(int)

merged = aba1_work.merge(aba2_rdr, on='id', how='left')
merged['INCOMING_DTTM'] = pd.to_datetime(merged['INCOMING_DTTM'], format='mixed')
merged['rdr_dt']        = pd.to_datetime(merged['rdr_dt'], format='mixed', errors='coerce')
merged['horas_ate_rdr'] = (merged['rdr_dt'] - merged['INCOMING_DTTM']).dt.total_seconds() / 3600
merged['abriu_rdr_48h'] = ((merged['horas_ate_rdr'] >= 0) & (merged['horas_ate_rdr'] <= 48)).astype(int)

agg = merged.groupby(['id', 'INCOMING_DTTM', 'CX_TEAM_NAME'])['abriu_rdr_48h'].max().reset_index()
print(f'Dataset: {agg.shape[0]:,} registros')
print(f'Taxa de conversão (target=1): {agg.abriu_rdr_48h.mean():.2%}')
print(agg['abriu_rdr_48h'].value_counts())

## 3 · Feature Engineering

In [ ]:
aba2_features = aba2[['ID do usuario', 'CX_PR_NAME', 'CDU_NAME', 'RDR_USER_DOC_TYPE', 'RDR_STATUS_NAME']].copy()
aba2_features.columns = ['id', 'cx_pr_name', 'cdu_name', 'doc_type', 'status']
aba2_features['id'] = pd.to_numeric(aba2_features['id'], errors='coerce')

df = agg.merge(aba2_features.drop_duplicates('id'), on='id', how='left')

df['dia_semana']    = df['INCOMING_DTTM'].dt.dayofweek
df['mes']           = df['INCOMING_DTTM'].dt.month
df['hora']          = df['INCOMING_DTTM'].dt.hour

reincidencia = df.groupby('id')['INCOMING_DTTM'].count().rename('n_contatos')
df = df.merge(reincidencia, on='id', how='left')
df['reincidente'] = (df['n_contatos'] > 1).astype(int)

df['canal_ivr']     = df['CX_TEAM_NAME'].str.contains('IVR', na=False).astype(int)
df['canal_offline'] = df['CX_TEAM_NAME'].str.contains('Offline', na=False).astype(int)
df['canal_rdr']     = df['CX_TEAM_NAME'].str.contains('RDR|Bacen', na=False).astype(int)

le = LabelEncoder()
for col in ['CX_TEAM_NAME', 'cx_pr_name', 'cdu_name', 'doc_type']:
    df[col] = df[col].fillna('Desconhecido')
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

df['produto_pix']    = df['cx_pr_name'].str.contains('Pix|PIX', na=False).astype(int)
df['produto_cartao'] = df['cx_pr_name'].str.contains('[Cc]art', na=False).astype(int)

FEATURES = [
    'CX_TEAM_NAME_enc', 'cx_pr_name_enc', 'cdu_name_enc', 'doc_type_enc',
    'dia_semana', 'mes', 'hora',
    'reincidente', 'n_contatos',
    'canal_ivr', 'canal_offline', 'canal_rdr',
    'produto_pix', 'produto_cartao',
]
TARGET = 'abriu_rdr_48h'

df_model = df[['id', 'CX_TEAM_NAME'] + FEATURES + [TARGET, 'INCOMING_DTTM']].dropna(subset=FEATURES)
print(f'Dataset final: {df_model.shape[0]:,} registros · {len(FEATURES)} features')
print(f'Taxa de eventos: {df_model[TARGET].mean():.2%}')

## 4 · Treinar GradientBoosting com validação temporal

In [ ]:
df_model = df_model.sort_values('INCOMING_DTTM').reset_index(drop=True)

corte   = pd.Timestamp('2025-11-01')
train   = df_model[df_model['INCOMING_DTTM'] < corte]
test    = df_model[df_model['INCOMING_DTTM'] >= corte]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Treino: {len(X_train):,} | Teste: {len(X_test):,}')
print(f'Positivos treino: {y_train.mean():.2%} | Positivos teste: {y_test.mean():.2%}')

modelo = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=SEED,
)
print('Treinando...')
modelo.fit(X_train, y_train)

probs = modelo.predict_proba(X_test)[:, 1]
auc   = roc_auc_score(y_test, probs)
print(f'\nAUC-ROC no teste (novembro): {auc:.4f}')

## 5 · Curva ROC e distribuição de scores

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('Taxa de Falsos Positivos')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos')
axes[0].set_title('Curva ROC — Modelo de Propensão RDR', fontweight='bold')
axes[0].legend()

axes[1].hist(probs[y_test == 0], bins=40, alpha=0.6, color='steelblue',
             label='Não abriu RDR', density=True)
axes[1].hist(probs[y_test == 1], bins=40, alpha=0.6, color='tomato',
             label='Abriu RDR (target)', density=True)
axes[1].axvline(0.70, color='black', linestyle='--', label='Threshold 0.70')
axes[1].set_xlabel('Score (probabilidade)')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Distribuição de Scores por Classe', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/02_roc_scores.png', bbox_inches='tight')
plt.show()

top10_idx = np.argsort(probs)[::-1][:int(len(probs) * 0.10)]
precision_top10 = y_test.iloc[top10_idx].mean()
print(f'Precision @ top-10%:        {precision_top10:.2%}')
print(f'Recall regulatório (≥0.70): {(probs[y_test == 1] >= 0.70).mean():.2%}')

## 6 · Explicabilidade com SHAP

In [ ]:
explainer   = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=10,
                  feature_names=FEATURES, show=False)
plt.title('SHAP — Importância das Features (Modelo Q2)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/02_shap_summary.png', bbox_inches='tight')
plt.show()

mean_shap    = np.abs(shap_values).mean(axis=0)
feat_importance = pd.DataFrame({'feature': FEATURES, 'shap_mean': mean_shap}) \
    .sort_values('shap_mean', ascending=False)
print(feat_importance.head(10).to_string(index=False))

## 7 · Score para toda a base e fila operacional

In [ ]:
df_model = df_model.copy()
all_probs = modelo.predict_proba(df_model[FEATURES])[:, 1]
df_model['score_risco'] = (all_probs * 100).round(1)

fila = (
    df_model[df_model['score_risco'] >= 70]
    [['id', 'CX_TEAM_NAME', 'INCOMING_DTTM', 'score_risco']]
    .sort_values('score_risco', ascending=False)
    .reset_index(drop=True)
)

print(f'Clientes na fila de interceptação (score ≥ 70): {len(fila):,}')
print(f'% da base: {len(fila)/len(df_model)*100:.1f}%')
print()
print(fila.head(10).to_string())

df_model[['id', 'INCOMING_DTTM', 'score_risco', TARGET]].to_csv('../outputs/02_scores.csv', index=False)
feat_importance.to_csv('../outputs/02_feature_importance.csv', index=False)
print('\nScores exportados para ../outputs/02_scores.csv')

## 8 · Conclusões do Q2

- **AUC-ROC** no conjunto de novembro demonstra boa discriminação entre clientes de alto e baixo risco.
- **Features mais impactantes (SHAP):** canal RDR/Bacen, produto Pix, reincidência e canal IVR.
- **Threshold 0.70** → fila de interceptação com alta precisão, evitando sobrecarga operacional.
- **Integração sugerida:** batch horário sobre novos acionamentos, resultado alimenta CRM com ação recomendada por agente.